In [16]:
!pip install pytorch_lightning torchmetrics catboost

In [17]:
from __future__ import annotations

import os
import zipfile
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import roc_auc_score
from catboost import CatBoostClassifier, Pool

In [18]:
data_dir: str = "/content"
train_csv: str = "train.csv"
test_csv: str = "test.csv"
submission_csv: str = "submission.csv"

id_col: str = "id"
target_col: str = "smoking"

seed: int = 42
val_size: float = 0.2

cat_unique_threshold: int = 20
force_cat_cols: tuple[str, ...] = ("Urine protein", "hearing(left)", "hearing(right)", "dental caries")

iterations: int = 5000
learning_rate: float = 0.03
depth: int = 8
l2_leaf_reg: float = 5.0
early_stopping_rounds: int = 200
verbose: int = 200
n_splits: int = 5

force_task_type: str = "CPU"

In [19]:
# добавляем несколько понятных производных признаков
def add_simple_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Very lightweight feature engineering (safe: only adds if columns exist).
    Helps tabular models a bit without making the code messy.
    """
    df = df.copy()

    # BMI = вес / рост^2 (рост в метрах)
    if "weight(kg)" in df.columns and "height(cm)" in df.columns:
        h_m = pd.to_numeric(df["height(cm)"], errors="coerce") / 100.0
        w = pd.to_numeric(df["weight(kg)"], errors="coerce")
        df["bmi"] = w / (h_m * h_m)

    # Талия / рост
    if "waist(cm)" in df.columns and "height(cm)" in df.columns:
        waist = pd.to_numeric(df["waist(cm)"], errors="coerce")
        height = pd.to_numeric(df["height(cm)"], errors="coerce")
        df["waist_to_height"] = waist / height

    # Производные от давления
    if "systolic" in df.columns and "relaxation" in df.columns:
        sys = pd.to_numeric(df["systolic"], errors="coerce")
        dia = pd.to_numeric(df["relaxation"], errors="coerce")
        df["pulse_pressure"] = sys - dia
        df["map"] = (2 * dia + sys) / 3.0

    # Соотношения липидов
    if "cholesterol" in df.columns and "HDL" in df.columns:
        chol = pd.to_numeric(df["cholesterol"], errors="coerce")
        hdl = pd.to_numeric(df["HDL"], errors="coerce")
        df["chol_to_hdl"] = chol / (hdl + 1e-6)
    if "HDL" in df.columns and "LDL" in df.columns:
        hdl = pd.to_numeric(df["HDL"], errors="coerce")
        ldl = pd.to_numeric(df["LDL"], errors="coerce")
        df["ldl_to_hdl"] = ldl / (hdl + 1e-6)
    if "triglyceride" in df.columns and "HDL" in df.columns:
        tg = pd.to_numeric(df["triglyceride"], errors="coerce")
        hdl = pd.to_numeric(df["HDL"], errors="coerce")
        df["tg_to_hdl"] = tg / (hdl + 1e-6)

    # Соотношение печёночных ферментов
    if "AST" in df.columns and "ALT" in df.columns:
        ast = pd.to_numeric(df["AST"], errors="coerce")
        alt = pd.to_numeric(df["ALT"], errors="coerce")
        df["ast_to_alt"] = ast / (alt + 1e-6)

    # Аггрегаты по глазам/ушам
    if "eyesight(left)" in df.columns and "eyesight(right)" in df.columns:
        el = pd.to_numeric(df["eyesight(left)"], errors="coerce")
        er = pd.to_numeric(df["eyesight(right)"], errors="coerce")
        df["eyesight_mean"] = (el + er) / 2.0
        df["eyesight_diff"] = (el - er).abs()
    if "hearing(left)" in df.columns and "hearing(right)" in df.columns:
        hl = pd.to_numeric(df["hearing(left)"], errors="coerce")
        hr = pd.to_numeric(df["hearing(right)"], errors="coerce")
        df["hearing_sum"] = hl + hr

    # Логи для “скошенных” значений
    for c in ["Gtp", "ALT", "AST", "triglyceride", "fasting blood sugar", "serum creatinine"]:
        if c in df.columns:
            v = pd.to_numeric(df[c], errors="coerce")
            df[f"log1p_{c}"] = np.log1p(v.clip(lower=0))

    return df

In [20]:
train_path = os.path.join(data_dir, train_csv)
test_path = os.path.join(data_dir, test_csv)
out_path = os.path.join(data_dir, submission_csv)

df = pd.read_csv(train_path)
df_test = pd.read_csv(test_path)

y = df[target_col].astype(int)
X = df.drop(columns=[target_col], errors="ignore")

if id_col in X.columns:
    X = X.drop(columns=[id_col])

X_test = df_test.copy()
if id_col in X_test.columns:
    X_test = X_test.drop(columns=[id_col])

X = add_simple_features(X)
X_test = add_simple_features(X_test)

In [21]:
# определяем, какие колонки считать категориальными по числу уникальных значений
def detect_cat_cols(X: pd.DataFrame, threshold: int) -> list[str]:
    cat_cols: list[str] = []
    for c in X.columns:
        nun = X[c].nunique(dropna=True)
        if nun <= threshold:
            cat_cols.append(c)
    return cat_cols

cat_cols = set(detect_cat_cols(X, cat_unique_threshold))
for c in force_cat_cols:
    if c in X.columns:
        cat_cols.add(c)
cat_cols = sorted(cat_cols)
cat_idx = [X.columns.get_loc(c) for c in cat_cols]

for c in cat_cols:
    X[c] = X[c].astype("string")
    X_test[c] = X_test[c].astype("string")

In [23]:
task_type = force_task_type
if task_type == "GPU" and not torch.cuda.is_available():
    task_type = "CPU"

# считаем коэффициент дисбаланса классов
pos = float(y.sum())
neg = float(len(y) - pos)
scale_pos_weight = neg / max(pos, 1.0)

# создаем фабрику моделей, чтобы на каждом фолде создавать новый CatBoost с теми же гиперпараметрами
def make_model(seed: int) -> "CatBoostClassifier":
    return CatBoostClassifier(
        loss_function="Logloss",
        eval_metric="AUC",
        iterations=iterations,
        learning_rate=learning_rate,
        depth=depth,
        l2_leaf_reg=l2_leaf_reg,
        random_strength=1.0,
        bootstrap_type="Bayesian",
        bagging_temperature=1.0,
        rsm=0.9,
        min_data_in_leaf=20,
        leaf_estimation_iterations=5,
        border_count=128,
        scale_pos_weight=scale_pos_weight,
        random_seed=seed,
        task_type=task_type,
        thread_count=-1 if task_type == "CPU" else None,
        verbose=verbose,
        allow_writing_files=False,
    )

In [24]:
# k-fold CV: делим train на 5 частей, обучаем 5 моделей и усредняем предсказания
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
oof = np.zeros(len(X), dtype=np.float32)
test_pred = np.zeros(len(X_test), dtype=np.float32)
fold_aucs: list[float] = []

for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), start=1):
    X_tr_f, y_tr_f = X.iloc[tr_idx], y.iloc[tr_idx]
    X_va_f, y_va_f = X.iloc[va_idx], y.iloc[va_idx]

    train_pool = Pool(X_tr_f, y_tr_f, cat_features=cat_idx if len(cat_idx) else None)
    val_pool = Pool(X_va_f, y_va_f, cat_features=cat_idx if len(cat_idx) else None)

    model = make_model(seed=seed + fold)
    model.fit(
        train_pool,
        eval_set=val_pool,
        use_best_model=True,
        early_stopping_rounds=early_stopping_rounds,
    )

    oof[va_idx] = model.predict_proba(X_va_f)[:, 1]
    fold_auc = roc_auc_score(y_va_f, oof[va_idx])
    fold_aucs.append(float(fold_auc))
    print(f"fold {fold}: val_auc={fold_auc:.6f}, best_iteration={model.get_best_iteration()}")

    test_pred += model.predict_proba(X_test)[:, 1] / n_splits

auc = roc_auc_score(y, oof)
print(f"OOF val_auc: {auc:.6f} (folds: {[round(a, 6) for a in fold_aucs]})")

0:	test: 0.8489419	best: 0.8489419 (0)	total: 82ms	remaining: 6m 50s
200:	test: 0.8795728	best: 0.8795728 (200)	total: 28s	remaining: 11m 8s
400:	test: 0.8820382	best: 0.8821435 (378)	total: 39.8s	remaining: 7m 36s
600:	test: 0.8833571	best: 0.8835196 (542)	total: 51.6s	remaining: 6m 17s
800:	test: 0.8832408	best: 0.8838465 (644)	total: 1m 3s	remaining: 5m 32s
Stopped by overfitting detector  (200 iterations wait)

bestTest = 0.8838464577
bestIteration = 644

Shrink model to first 645 iterations.
fold 1: val_auc=0.883846, best_iteration=644
0:	test: 0.8568700	best: 0.8568700 (0)	total: 56.1ms	remaining: 4m 40s
200:	test: 0.8881458	best: 0.8882592 (199)	total: 11.4s	remaining: 4m 32s
400:	test: 0.8895448	best: 0.8898593 (352)	total: 23.2s	remaining: 4m 25s
600:	test: 0.8893351	best: 0.8901728 (460)	total: 35s	remaining: 4m 16s
Stopped by overfitting detector  (200 iterations wait)

bestTest = 0.8901728182
bestIteration = 460

Shrink model to first 461 iterations.
fold 2: val_auc=0.89017

In [25]:
sub = pd.DataFrame({id_col: df_test[id_col].values, target_col: test_pred.astype(float)})
sub.to_csv(out_path, index=False)
print("Saved:", out_path)
print(sub.head())

Saved: /content/submission.csv
      id   smoking
0  15000  0.317625
1  15001  0.070025
2  15002  0.076824
3  15003  0.372631
4  15004  0.020131
